# Module and function import

Import modules and simple plotter.

In [ ]:
from pylab import *
import matplotlib.pyplot as plt
from matplotlib import ticker, cm
from matplotlib import colors
from scipy import integrate
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 unused import
from scipy import interpolate
%matplotlib inline

In [ ]:
from scipy.interpolate import InterpolatedUnivariateSpline

In [ ]:
plt.rcParams.update({'font.size': 12})

Simple 2D plotter.

In [ ]:
def plot_2d(n, r_ex, z_ex, u, uname):
    if n == 1:
        fig, ax = plt.subplots(figsize = (8, 8))
        im = ax.imshow(u.T, origin = 'lower', cmap = cm.viridis, extent = [r_ex[0], r_ex[1], z_ex[0], z_ex[1]])
        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$z$")
        ax.set_title(uname)
        fig.colorbar(im, fraction=0.046, pad=0.04, ax = ax)
        
    if n > 1:
        fig = plt.figure(figsize = (16, ((n + 1) // 2) * 8))
        
        ax = []
        
        for i in range(n):
            ax.append( fig.add_subplot((n + 1) // 2, 2, i + 1))
            im = ax[-1].imshow(u[i].T, origin = 'lower', cmap = cm.viridis, extent = [r_ex[0], r_ex[1], z_ex[0], z_ex[1]])
            ax[-1].set_xlabel(r"$\rho$")
            ax[-1].set_ylabel(r"$z$")
            ax[-1].set_title(uname[i])
            plt.colorbar(im, fraction=0.046, pad=0.04, ax = ax[-1])
    
    plt.show()   

# Simulation Parameters

Output directory name.

In [ ]:
# dir_name = "/mnt/e/RBS/Catalogue2/l=6,w=5.97232E-01,dr=4.00000E-02,N=0800/"
dir_name = "Catalogue2/l=6,w=5.97232E-01,dr=4.00000E-02,N=0800/"

Rotation number, finite difference order (and ghost zones), $\omega$ and scalar field mass $m$.

In [ ]:
l = 6
order = 4
ghost = 2
m = 1.0

Coordinate grids $(\rho,\,z)$.

In [ ]:
r = np.genfromtxt(dir_name + "r.asc")
z = np.genfromtxt(dir_name + "z.asc")
rr = np.sqrt(r**2 + z**2)

Step sizes.

In [ ]:
dr = r[1,0] - r[0,0]
dz = z[0,1] - z[0,0]
print(dr, dz)

Finite difference error order and boundary approximation error.

In [ ]:
np.sqrt(dr*dz)**order, r[-1,-1]**(-2.0)

Grid extensions.

In [ ]:
r_min, r_max = np.min(r), np.max(r)
z_min, z_max = np.min(z), np.max(z)
r_ex = [r_min, r_max]
z_ex = [z_min, z_max]
print(r_min, r_max)
print(z_min, z_max)

# Initial Data

Initial data.

In [ ]:
log_alpha_i = np.genfromtxt(dir_name + "log_alpha_i.asc")
log_h_i     = np.genfromtxt(dir_name + "log_h_i.asc")
log_a_i     = np.genfromtxt(dir_name + "log_a_i.asc")
psi_i       = np.genfromtxt(dir_name + "psi_i.asc")
beta_i      = np.genfromtxt(dir_name + "beta_i.asc")
lambda_i    = np.genfromtxt(dir_name + "lambda_i.asc")

In [ ]:
w_i = np.genfromtxt(dir_name + "w_i.asc").item()

In [ ]:
w = np.arctanh(2.0 * w_i / m - 1.0)

Metric variables.

In [ ]:
plot_2d(4, r_ex, z_ex, [log_alpha_i, beta_i, log_h_i, log_a_i] , [r"$\log\,\alpha_i$", r"$\beta_i$", r"$\log\,h_i$", r"$\log\,a_i$"])

In [ ]:
plot_2d(4, r_ex, z_ex, [np.exp(log_alpha_i), beta_i, np.exp(log_h_i), np.exp(log_a_i)] , [r"$\alpha_i$", r"$\beta_i$", r"$h_i$", r"$a_i$"])

Regularization variable.

In [ ]:
plot_2d(1, r_ex, z_ex, lambda_i, r"$(\lambda)_i$")

Scalar field and associated variable.

In [ ]:
plot_2d(2, r_ex, z_ex, [(r**l*psi_i)[:,:], (psi_i)[:,:]], [r"$\phi_i$", r"$\psi_i = \rho^{-l}\,\phi_i$"])

View scalar field on equator and from maximum peak parallel to axis.

In [ ]:
fig, ax = plt.subplots(figsize = (14, 6), ncols = 2)
ax[0].plot(r[:64,ghost], (psi_i)[:64,ghost], ".-")
ax[0].set_xlabel(r"$\rho$")
ax[0].set_ylabel(r"$\phi$")
ax[0].set_title(r"Scalar field $\psi_i$ profile on equator")

k = np.argmax((psi_i * r**l)[:,ghost])

ax[1].plot(z[k,:64], (psi_i)[k,:64])
ax[1].set_xlabel(r"$z$")
ax[1].set_ylabel(r"$\phi$")
ax[1].set_title(r"Scalar field $\psi_i$ profile on $\rho = %5.3lf$" % r[k, k])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (14, 6), ncols = 2)
ax[0].plot(r[:64,ghost], (psi_i * r**l)[:64,ghost], ".-")
ax[0].set_xlabel(r"$\rho$")
ax[0].set_ylabel(r"$\phi$")
ax[0].set_title(r"Scalar field $\phi_i$ profile on equator")

k = np.argmax((psi_i * r**l)[:,ghost])

ax[1].plot(z[k,:64], (psi_i * r**l)[k,:64])
ax[1].set_xlabel(r"$z$")
ax[1].set_ylabel(r"$\phi$")
ax[1].set_title(r"Scalar field $\phi_i$ profile on $\rho = %5.3lf$" % r[k, k])
plt.show()

In [ ]:
print(k, r[k,0])

Initial RHS's.

In [ ]:
f1_i = np.genfromtxt(dir_name + "f0_i.asc")
f2_i = np.genfromtxt(dir_name + "f1_i.asc")
f3_i = np.genfromtxt(dir_name + "f2_i.asc")
f4_i = np.genfromtxt(dir_name + "f3_i.asc")
f5_i = np.genfromtxt(dir_name + "f4_i.asc")
f6_i = np.genfromtxt(dir_name + "f5_i.asc")

Metric variables.

In [ ]:
plot_2d(4, r_ex, z_ex, [f1_i, f2_i, f3_i, f4_i] , [r"$(f_1)_i$", r"$(f_2)_i$", r"$(f_3)_i$", r"$(f_4)_i$"])

Scalar field initial RHS.

In [ ]:
plot_2d(2, r_ex, z_ex, [f5_i, f6_i] , [r"$(f_5)_i$", r"$(f_6)_i$"])

# Newton-Raphson Final Iteration

Variables.

In [ ]:
log_alpha_f = np.genfromtxt(dir_name + "log_alpha_f.asc")
log_h_f     = np.genfromtxt(dir_name + "log_h_f.asc")
log_a_f     = np.genfromtxt(dir_name + "log_a_f.asc")
psi_f       = np.genfromtxt(dir_name + "psi_f.asc")
beta_f      = np.genfromtxt(dir_name + "beta_f.asc")
lambda_f    = np.genfromtxt(dir_name + "lambda_f.asc")

In [ ]:
w = w_f = np.genfromtxt(dir_name + "w_f.asc").item()

In [ ]:
plot_2d(4, r_ex, z_ex, [log_alpha_f, beta_f, log_h_f, log_a_f] , [r"$\log\,\alpha_f$", r"$\beta_f$", r"$\log\,h_f$", r"$\log\,a_f$"])

In [ ]:
plot_2d(2, r_ex, z_ex, [psi_f, lambda_f] , [r"$\psi_f$", r"$\lambda_f$"])

In [ ]:
np.any(lambda_f > 0.0)

In [ ]:
plot_2d(4, r_ex, z_ex, [(log_alpha_f - log_alpha_i) / log_alpha_f, (beta_f - beta_i) / beta_f, (log_h_f - log_h_i) / log_h_f, (log_a_f - log_a_i) / log_a_f] , [r"$\Delta_R\,\log\,\alpha$", r"$\Delta_R\,\beta_f$", r"$\Delta_R\,\log\,h$", r"$\Delta_R\,\log\,a$"])

In [ ]:
plot_2d(2, r_ex, z_ex, [(psi_f - psi_i) / psi_f, (lambda_f - lambda_i) / lambda_f] , [r"$\Delta_R\,\psi$", r"$\Delta_R\,\lambda$"])

In [ ]:
#dl = 0.1
#np.savetxt(dir_name + "log_alpha_ig.asc", log_alpha_f + dl * (log_alpha_f - log_alpha_i))
#np.savetxt(dir_name + "beta_ig.asc", beta_f + dl * (beta_f - beta_i))
#np.savetxt(dir_name + "log_h_ig.asc", log_h_f + dl * (log_h_f - log_h_i))
#np.savetxt(dir_name + "log_a_ig.asc", log_a_f + dl * (log_a_f - log_a_i))
#np.savetxt(dir_name + "psi_ig.asc", psi_f + dl * (psi_f - psi_i))
#np.savetxt(dir_name + "lambda_ig.asc", lambda_f + dl * (lambda_f - lambda_i))

In [ ]:
max_phi = np.genfromtxt(dir_name + "phi_max.asc").item()

In [ ]:
k

In [ ]:
fig, ax = plt.subplots(figsize = (14, 6), ncols = 2)
ax[0].plot(r[:256,ghost], (r**l * psi_f)[:256,ghost], ".-", label = r"Final")
ax[0].plot()
ax[0].set_xlabel(r"$\rho$")
ax[0].set_ylabel(r"$\phi$")
ax[0].set_title(r"Scalar field $\phi_f$ profile on equator")
ax[0].legend()

k = np.argmax((psi_f * r**l)[:,ghost])

ax[1].plot(z[k,:256], (r**l * psi_f)[k,:256], label = r"Final")
ax[1].set_xlabel(r"$z$")
ax[1].set_ylabel(r"$\phi$")
ax[1].set_title(r"Scalar field $\phi_f$ profile on $\rho = %5.3lf$" % r[k, k])
ax[1].legend()
plt.show()

Final RHS's.

In [ ]:
f1_f = np.genfromtxt(dir_name + "f0_f.asc")
f2_f = np.genfromtxt(dir_name + "f1_f.asc")
f3_f = np.genfromtxt(dir_name + "f2_f.asc")
f4_f = np.genfromtxt(dir_name + "f3_f.asc")
f5_f = np.genfromtxt(dir_name + "f4_f.asc")
f6_f = np.genfromtxt(dir_name + "f5_f.asc")

In [ ]:
plot_2d(4, r_ex, z_ex, [f1_f, f2_f, f3_f, f4_f] , [r"$(f_1)_f$", r"$(f_2)_f$", r"$(f_3)_f$", r"$(f_4)_f$"])

In [ ]:
plot_2d(2, r_ex, z_ex, [f5_f, f6_f], [r"$(f_5)_f$", r"$(f_6)_f$"])

In [ ]:
du1_f = np.genfromtxt(dir_name + "du0_f.asc")
du2_f = np.genfromtxt(dir_name + "du1_f.asc")
du3_f = np.genfromtxt(dir_name + "du2_f.asc")
du4_f = np.genfromtxt(dir_name + "du3_f.asc")
du5_f = np.genfromtxt(dir_name + "du4_f.asc")
du6_f = np.genfromtxt(dir_name + "du5_f.asc")

In [ ]:
plot_2d(4, r_ex, z_ex, [du1_f, du2_f, du3_f, du4_f] , [r"$(\Delta u_1)_f$", r"$(\Delta u_2)_f$", r"$(\Delta u_3)_f$", r"$(\Delta u_4)_f$"])

In [ ]:
plot_2d(2, r_ex, z_ex, [du5_f[ghost:,ghost:], du6_f[ghost:,ghost:]], [r"$(\Delta u_5)_f$", r"$(\Delta u_6)_f$"])

## Initial Data Minus Final Data

In [ ]:
w_i, w_f

In [ ]:
initial_peaks = np.array([np.max(np.abs(log_alpha_i)), np.max(np.abs(beta_i)), np.max(np.abs(log_h_i)), np.max(np.abs(log_a_i)), np.max(np.abs(psi_i)), np.max(np.abs(lambda_i)), 1.0 / w_i])

In [ ]:
final_peaks = np.array([np.max(np.abs(log_alpha_f)), np.max(np.abs(beta_f)), np.max(np.abs(log_h_f)), np.max(np.abs(log_a_f)), np.max(np.abs(psi_f)), np.max(np.abs(lambda_f)), 1.0 / w_f])

In [ ]:
final_peaks / initial_peaks

In [ ]:
plot_2d(4, r_ex, z_ex, [log_alpha_f / log_alpha_i, beta_f / beta_i, log_h_f / log_h_i, log_a_f / log_a_i] , [r"$\frac{\log\,\alpha_f}{\log\,\alpha_i}$", r"$\frac{\beta_f}{\beta_i}$", r"$\frac{\log\,h_f}{\log\,h_i}$", r"$\frac{\log\,a_f}{\log\,a_i}$"])

In [ ]:
plot_2d(2, r_ex, z_ex, [psi_f / psi_i, lambda_f / lambda_i] , [r"$\frac{\psi_f}{\psi_i}$", r"$\frac{\lambda_f}{\lambda_i}$"])

# Full Quadrant Plots

Contour plot over all four quadrants.

In [ ]:
def fill_true_variable(u, r_sym=1, z_sym=1):
    n_i = u.shape[0] - ghost
    n_j = u.shape[1] - ghost
    t_u = np.zeros((2 * n_i, 2 * n_j))
    t_u[n_i:,n_j:] = u[ghost:,ghost:]
    t_u[:n_i,:n_j] = r_sym * z_sym * u[-1:ghost-1:-1,-1:ghost-1:-1]
    t_u[:n_i,n_j:] = r_sym * u[-1:ghost-1:-1,ghost:]
    t_u[n_i:,:n_j] = z_sym * u[ghost:,-1:ghost-1:-1]
    return t_u

In [ ]:
t_r = fill_true_variable(r, -1, +1)
t_z = fill_true_variable(z, +1, -1)

t_log_alpha = fill_true_variable(log_alpha_f)
t_beta      = fill_true_variable(beta_f)
t_log_a     = fill_true_variable(log_a_f)
t_log_h     = fill_true_variable(log_h_f)
t_psi       = fill_true_variable(psi_f)

In [ ]:
t_lambda = fill_true_variable(lambda_f)

In [ ]:
%matplotlib inline

In [ ]:
plt.rcParams.update({'font.size': 12})

In [ ]:
# Figure initialization.
fig, ax = plt.subplots(figsize = (16,18), ncols=2, nrows=3)

plt.suptitle(r"Contour Plots For Physical Variables With $l = %d$, $\omega = %.3f (m/\hbar)$" % (l, w_f))

# Alpha.
v = np.exp(t_log_alpha)
vmax = v.max()
vmin = v.min()
n_levels = 21

norm = cm.colors.LogNorm(vmax = vmax, vmin = vmin)
cmap = cm.tab20b
extent = [t_r[0,0], t_r[-1,0], t_z[0,0], t_z[0,-1]]
levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)

im = ax[0][0].imshow(v.T, cmap = cmap, extent = extent, norm = norm, interpolation='spline36')
cnt = ax[0][0].contour(t_r, t_z, v, levels = levels, cmap=cmap, norm = norm, linestyles='solid', linewidths=1.5)

ax[0][0].set_xlabel(r"$\rho\, [\hbar/m]$")
ax[0][0].set_ylabel(r"$z\, [\hbar/m]$")
ax[0][0].set_title(r"Contour Levels Lapse $\alpha$")

#ax[0][0].set_xlim(-50,50)
#ax[0][0].set_ylim(-50,50)

cbar = fig.colorbar(im, fraction=0.046, pad=0.04, ax = ax[0][0], format='%.1e', ticks=levels)


# Beta.
v = np.abs(t_beta)
vmax = v.max()
vmin = v.min()
n_levels = 21

norm = cm.colors.LogNorm(vmax = vmax, vmin = vmin)
cmap = cm.tab20b
extent = [t_r[0,0], t_r[-1,0], t_z[0,0], t_z[0,-1]]
levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)

im = ax[0][1].imshow(v.T, cmap = cmap, extent = extent, norm = norm, interpolation='spline36')
cnt = ax[0][1].contour(t_r, t_z, v, levels = levels, cmap=cmap, norm = norm, linestyles='solid', linewidths=1.5)

ax[0][1].set_xlabel(r"$\rho\, [\hbar/m]$")
ax[0][1].set_ylabel(r"$z\, [\hbar/m]$")
ax[0][1].set_title(r"Contour Levels Shift $\left|\Omega\right|$")

#ax[0][1].set_xlim(-50,50)
#ax[0][1].set_ylim(-50,50)

cbar = fig.colorbar(im, fraction=0.046, pad=0.04, ax = ax[0][1], format='%.1e', ticks=levels)


# H.
v = np.exp(2.0 * t_log_h)
vmax = v.max()
vmin = v.min()
n_levels = 21

norm = cm.colors.LogNorm(vmax = vmax, vmin = vmin)
cmap = cm.tab20b
extent = [t_r[0,0], t_r[-1,0], t_z[0,0], t_z[0,-1]]
levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)

im = ax[1][0].imshow(v.T, cmap = cmap, extent = extent, norm = norm, interpolation='spline36')
cnt = ax[1][0].contour(t_r, t_z, v, levels = levels, cmap=cmap, norm = norm, linestyles='solid', linewidths=1.5)

ax[1][0].set_xlabel(r"$\rho\, [\hbar/m]$")
ax[1][0].set_ylabel(r"$z\, [\hbar/m]$")
ax[1][0].set_title(r"Contour Levels Metric $H = \gamma_{\varphi \varphi} / \rho^2$")

#ax[1][0].set_xlim(-50,50)
#ax[1][0].set_ylim(-50,50)

cbar = fig.colorbar(im, fraction=0.046, pad=0.04, ax = ax[1][0], format='%.1e', ticks=levels)


# A.
v = np.exp(2.0 * t_log_a)
vmax = v.max()
vmin = v.min()
n_levels = 21

norm = cm.colors.LogNorm(vmax = vmax, vmin = vmin)
cmap = cm.tab20b
extent = [t_r[0,0], t_r[-1,0], t_z[0,0], t_z[0,-1]]
levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)

im = ax[1][1].imshow(v.T, cmap = cmap, extent = extent, norm = norm, interpolation='spline36')
cnt = ax[1][1].contour(t_r, t_z, v, levels = levels, cmap=cmap, norm = norm, linestyles='solid', linewidths=1.5)

ax[1][1].set_xlabel(r"$\rho\, [\hbar/m]$")
ax[1][1].set_ylabel(r"$z\, [\hbar/m]$")
ax[1][1].set_title(r"Contour Levels Metric $A = \gamma_{rr}$")

#ax[1][1].set_xlim(-50,50)
#ax[1][1].set_ylim(-50,50)

cbar = fig.colorbar(im, fraction=0.046, pad=0.04, ax = ax[1][1], format='%.1e', ticks=levels)

# Phi
v = t_r**l * t_psi
vmax = v.max()
vmin = 1e-16
n_levels = 21

norm = cm.colors.LogNorm(vmax = vmax, vmin = vmin)
cmap = cm.tab20b
extent = [t_r[0,0], t_r[-1,0], t_z[0,0], t_z[0,-1]]
levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)

im = ax[2][0].imshow(v.T, cmap = cmap, extent = extent, norm = norm)
cnt = ax[2][0].contour(t_r, t_z, v, levels = levels, cmap=cmap, norm = norm, linestyles='solid', linewidths=1.5)

ax[2][0].set_xlabel(r"$\rho\, [\hbar/m]$")
ax[2][0].set_ylabel(r"$z\, [\hbar/m]$")
ax[2][0].set_title(r"Contour Levels $\phi$")

#ax[2][0].set_xlim(-50,50)
#ax[2][0].set_ylim(-50,50)

cbar = fig.colorbar(im, fraction=0.046, pad=0.04, ax = ax[2][0], format='%.1e', ticks=levels)

# Lambda.
v = np.abs(t_lambda)
vmax = v.max()
vmin = v.min()
n_levels = 21

norm = cm.colors.LogNorm(vmax = vmax, vmin = vmin)
cmap = cm.tab20b
extent = [t_r[0,0], t_r[-1,0], t_z[0,0], t_z[0,-1]]
levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)

im = ax[2][1].imshow(v.T, cmap = cmap, extent = extent, norm = norm, interpolation='spline36')
cnt = ax[2][1].contour(t_r, t_z, v, levels = levels, cmap=cmap, norm = norm, linestyles='solid', linewidths=1.5)

ax[2][1].set_xlabel(r"$\rho\, [\hbar/m]$")
ax[2][1].set_ylabel(r"$z\, [\hbar/m]$")
ax[2][1].set_title(r"Contour Levels Regularization $|\lambda|$")

#ax[2][1].set_xlim(-50,50)
#ax[2][1].set_ylim(-50,50)

cbar = fig.colorbar(im, fraction=0.046, pad=0.04, ax = ax[2][1], format='%.1e', ticks=levels)

fig.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (10, 10))

cnt = ax.contour(t_r, t_z, np.abs(t_r**l * t_psi), np.logspace(np.log10(np.min(np.abs(t_r**l * t_psi))), np.log10(np.max(np.abs(t_r**l * t_psi))), 30), norm=colors.LogNorm())

ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$z$")
ax.set_title(r"Contour Levels $\phi$")

#ax.set_xlim(-50,50)
#ax.set_ylim(-50,50)

fig.colorbar(cnt, fraction=0.046, pad=0.04, ax = ax, format='%.0e')

plt.show() 

# Gaussian Fit for Scalar Field

The aim is to fit 

$$\phi \approx \rho^l \psi_0\,\exp\left(-\frac{\rho^2}{2\sigma_\rho^2}\right)\,\exp\left(-\frac{z^2}{2\sigma_z^2}\right)\,.$$

In [ ]:
# First print out rho value where phi is maximum and its value there.
k = np.argmax((psi_f * r**l)[:,ghost])
k, r[k,ghost], psi_f[k,ghost] * r[k,ghost]**l
# We can calculate the width simply because in this Gaussian model, the maximum value
# of phi occurs at sqrt(l) * sigma_r.
sigma_r = r[k,ghost] / np.sqrt(l)
# The overall multiplication constant can now be calculated.
psi0 = (psi_f[k,ghost] * r[k,ghost]**l) / (r[k,ghost]**l * np.exp(-l/2.0))
# To get the width in z, we integrate
sigma_z = 2.0 * integrate.simps(( psi_f * r**l)[k,:], z[k,:]) / (np.sqrt(2.0 * np.pi) * psi0 * np.exp(-0.5 * l) * (np.sqrt(l) * sigma_r)**l)

In [ ]:
print(psi0, sigma_r, sigma_z)

In [ ]:
fig, ax = plt.subplots(figsize = (16, 5), ncols=2)
ax[0].plot(r[ghost:,ghost], (psi_f * r**l)[ghost:,ghost], label = r"Final $\phi$")
ax[0].plot(r[ghost:,ghost], (psi0 * r**l * np.exp(-0.5 * r**2 / sigma_r**2) * np.exp(-0.5 * z**2 / sigma_z**2))[ghost:,ghost], label = r"Gaussian Fit")
ax[0].set_xlabel(r"$\rho$")
ax[0].set_ylabel(r"$\phi$")
ax[0].legend(loc='upper right')
ax[0].set_title(r"Scalar field $\phi_f$ profile on equator")

k = np.argmax((psi_f * r**l)[:,ghost])

ax[1].plot(z[k,ghost:], (psi_f * r**l)[k,ghost:], label = r"Final $\phi$")
ax[1].plot(z[k,ghost:], (psi0 * r**l * np.exp(-0.5 * r**2 / sigma_r**2) * np.exp(-0.5 * z**2 / sigma_z**2))[k,ghost:], label = r"Gaussian Fit")
ax[1].set_xlabel(r"$z$")
ax[1].set_ylabel(r"$\phi$")
ax[1].legend(loc='upper right')
ax[1].set_title(r"Scalar field $\phi_f$ profile on $\rho = %5.3lf$" % r[k, k])
plt.show()

# Analysis

Derivative subroutines.

In [ ]:
# Required quantities: dr, dz. 

# Differentiate with respect to r (first variable/index slot). 
# Input variable is u an even (uneven) function of r with r_sym = 1 (r_sym = -1).
# Will use centered finite differences of order = 4 by default, 
# although order = 2 is also supported.
def Dr_u(u, order=4, r_sym=1):
    # Zero block like variable to differentiate.
    d = np.zeros_like(u)
    # Fourth order.
    if order == 4:
        # Centered stencil.
        d[2:-2,:] = (1.0 / 12.0) * (-u[4:,:] + 8.0 * u[3:-1,:] - 8.0 * u[1:-3,:] + u[:-4,:]) / dr
        # Parity ghost zones.
        d[1,:] = -r_sym * d[2,:]
        d[0,:] = -r_sym * d[3,:]
        # Boundaries use (-3,-2,-1,0,1) and (-4,-3,-2,-1,0) stencils.
        d[-2,:] = (1.0 / 12.0) * ( 3.0 * u[-1,:] + 10.0 * u[-2,:] - 18.0 * u[-3,:] +  6.0 * u[-4,:] -       u[-5,:]) / dr
        d[-1,:] = (1.0 / 12.0) * (25.0 * u[-1,:] - 48.0 * u[-2,:] + 36.0 * u[-3,:] - 16.0 * u[-4,:] + 3.0 * u[-5,:]) / dr
    # 2nd order.
    elif order == 2:
        d[1:-1,:] = 0.5 * (u[2:,:] - u[:-2,:]) / dr
        # Parity ghost zones.
        d[0,:] = -r_sym * d[1,:]
        # (-2,-1,0) stencil.
        d[-1,:] = 0.5 * (3.0 * u[-1,:] - 4.0 * u[-2,:] + u[-3,:]) / dr
    return d
    
# Same, but for z (second variable/index slot).
def Dz_u(u, order=4, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[:,2:-2] = (1.0 / 12.0) * (-u[:,4:] + 8.0 * u[:,3:-1] - 8.0 * u[:,1:-3] + u[:,:-4]) / dz
        d[:,1] = -z_sym * d[:,2]
        d[:,0] = -z_sym * d[:,3]
        d[:,-2] = (1.0 / 12.0) * ( 3.0 * u[:,-1] + 10.0 * u[:,-2] - 18.0 * u[:,-3] +  6.0 * u[:,-4] -       u[:,-5]) / dz
        d[:,-1] = (1.0 / 12.0) * (25.0 * u[:,-1] - 48.0 * u[:,-2] + 36.0 * u[:,-3] - 16.0 * u[:,-4] + 3.0 * u[:,-5]) / dz
    elif order == 2:
        d[:,1:-1] = 0.5 * (u[2:,:] - u[:-2,:]) / dz
        d[:,0] = -z_sym * d[:,1]
        d[:,-1] = 0.5 * (3.0 * u[:,-1] - 4.0 * u[:,-2] + u[:,-3]) / dz
    return d

# Second derivative.
def Drr_u(u, order=4, r_sym=1):
    d = np.zeros_like(u)
    # Fourth order.
    if order == 4:
        # Centered stencil.
        d[2:-2,:] = - (1.0 / 12.0) * (30.0 * u[2:-2,:] - 16.0 * (u[3:-1,:] + u[1:-3,:]) + (u[4:,:] + u[:-4,:])) / dr**2
        # Parity ghost zones.
        d[1,:] = r_sym * d[2,:]
        d[0,:] = r_sym * d[3,:]
        # Boundaries use (-4,-3,-2,-1,0,1) and (-5,-4,-3,-2,-1,0) stencils.
        d[-2,:] = (1.0 / 12.0) * (10.0 * u[-1,:] -  15.0 * u[-2,:] -   4.0 * u[-3,:] +  14.0 * u[-4,:] -  6.0 * u[-5,:] +        u[-6,:]) / dr**2
        d[-1,:] = (1.0 / 12.0) * (45.0 * u[-1,:] - 154.0 * u[-2,:] + 214.0 * u[-3,:] - 156.0 * u[-4,:] + 61.0 * u[-5,:] - 10.0 * u[-6,:]) / dr**2
    # 2nd order.
    elif order == 2:
        d[1:-1,:] = (u[2:,:] - 2.0 * u[1:-1,:] + u[:-2,:]) / dr**2
        # Parity ghost zones.
        d[0,:] = r_sym * d[1,:]
        # (-3,-2,-1,0) stencil.
        d[-1,:] = (2.0 * u[-1,:] - 5.0 * u[-2,:] + 4.0 * u[-3,:] - u[-4,:]) / dr**2
    return d
    
# Same, but for z.
def Dzz_u(u, order=4, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[:,2:-2] = - (1.0 / 12.0) * (30.0 * u[:,2:-2] - 16.0 * (u[:,3:-1] + u[:,1:-3]) + (u[:,4:] + u[:,:-4])) / dz**2
        d[:,1] = z_sym * d[:,2]
        d[:,0] = z_sym * d[:,3]
        d[:,-2] = (1.0 / 12.0) * (10.0 * u[:,-1] -  15.0 * u[:,-2] -   4.0 * u[:,-3] +  14.0 * u[:,-4] -  6.0 * u[:,-5] +        u[:,-6]) / dz**2
        d[:,-1] = (1.0 / 12.0) * (45.0 * u[:,-1] - 154.0 * u[:,-2] + 214.0 * u[:,-3] - 156.0 * u[:,-4] + 61.0 * u[:,-5] - 10.0 * u[:,-6]) / dz**2
    elif order == 2:
        d[:,1:-1] = (u[:,2:] - 2.0 * u[:,1:-1] + u[:,:-2]) / dz**2
        d[:,0] = z_sym * d[:,1]
        d[:,-1] = (2.0 * u[:-1] - 5.0 * u[:-2] + 4.0 * u[:-3] - u[:-4]) / dz**2
    return d

In [ ]:
def one_dim_Drr_u(u, order=4, rr_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[2:-2] = (1.0 / 12.0) * (-u[4:] + 8.0 * u[3:-1] - 8.0 * u[1:-3] + u[:-4]) / drr
        d[1] = (1.0 / 12.0) * (-u[3] + 8.0 * u[2] + rr_sym * u[1] - 8.0 * u[0]) / drr
        d[0] = (1.0 / 12.0) * (-u[2] * (1.0 - rr_sym) + 8.0 * u[1] * (1.0 - rr_sym)) / drr
        d[-2] = (1.0 / 12.0) * (3.0 * u[-1] + 10.0 * u[-2] - 18.0 * u[-3] + 6.0 * u[-4] - u[-5]) / drr
        d[-1] = (1.0 / 12.0) * (25.0 * u[-1] - 48.0 * u[-2] + 36.0 * u[-3] - 16.0 * u[-4] + 3.0 * u[-5]) / drr
    elif order == 2:
        d[1:-1] = 0.5 * (u[2:] - u[:-2]) / drr
        d[0] = 0.5 * u[1] * (1.0 - rr_sym) / drr
        d[-1] = 0.5 * (3.0 * u[-1] - 4.0 * u[-2] + u[-3]) / drr
    return d

def one_dim_Dth_u(u, order=4, r_sym=1, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[2:-2] = (1.0 / 12.0) * (-u[4:] + 8.0 * u[3:-1] - 8.0 * u[1:-3] + u[:-4]) / dth
        d[0] = (1.0 / 12.0) * (-u[2] + 8.0 * u[1]) * (1.0 - r_sym) / dth
        d[1] = (1.0 / 12.0) * (-8.0 * u[0] + r_sym * u[1] + 8.0 * u[2] - u[3]) / dth
        d[-2] = -(1.0 / (12.0 * dth)) * (-8.0 * u[-1] + z_sym * u[-2] + 8.0 * u[-3] - u[-4])
        d[-1] = -(1.0 / (12.0 * dth)) * (-u[-3] + 8.0 * u[-2]) * (1.0 - z_sym)
    elif order == 2:
        d[1:-1] = 0.5 * (u[2:] - u[:-2]) / dth
        d[0] = 0.5 * u[1] * (1.0 - r_sym) / dth
        d[-1] = -0.5 * u[-1] * (1.0 - z_sym) / dth
    return d

def two_dim_Drr_u(u, order=4, rr_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[2:-2,:] = (1.0 / 12.0) * (-u[4:,:] + 8.0 * u[3:-1,:] - 8.0 * u[1:-3,:] + u[:-4,:]) / drr
        d[1,:] = (1.0 / 12.0) * (-u[3,:] + 8.0 * u[2,:] + rr_sym * u[1,:] - 8.0 * u[0,:]) / drr
        d[0,:] = (1.0 / 12.0) * (-u[2,:] * (1.0 - rr_sym) + 8.0 * u[1,:] * (1.0 - rr_sym)) / drr
        d[-2,:] = (1.0 / 12.0) * (3.0 * u[-1,:] + 10.0 * u[-2,:] - 18.0 * u[-3,:] + 6.0 * u[-4,:] - u[-5,:]) / drr
        d[-1,:] = (1.0 / 12.0) * (25.0 * u[-1,:] - 48.0 * u[-2,:] + 36.0 * u[-3,:] - 16.0 * u[-4,:] + 3.0 * u[-5,:]) / drr
    elif order == 2:
        d[1:-1,:] = 0.5 * (u[2:,:] - u[:-2,:]) / drr
        d[0,:] = 0.5 * u[1,:] * (1.0 - rr_sym) / drr
        d[-1,:] = 0.5 * (3.0 * u[-1,:] - 4.0 * u[-2,:] + u[-3,:]) / drr
    return d

def two_dim_Dth_u(u, order=4, r_sym=1, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[:,2:-2] = (1.0 / 12.0) * (-u[:,4:] + 8.0 * u[:,3:-1] - 8.0 * u[:,1:-3] + u[:,:-4]) / dth
        d[:,0] = (1.0 / 12.0) * (-u[:,2] + 8.0 * u[:,1]) * (1.0 - r_sym) / dth
        d[:,1] = (1.0 / 12.0) * (-8.0 * u[:,0] + r_sym * u[:,1] + 8.0 * u[:,2] - u[:,3]) / dth
        d[:,-2] = -(1.0 / (12.0 * dth)) * (-8.0 * u[:,-1] + z_sym * u[:,-2] + 8.0 * u[:,-3] - u[:,-4])
        d[:,-1] = -(1.0 / (12.0 * dth)) * (-u[:,-3] + 8.0 * u[:,-2]) * (1.0 - z_sym)
    elif order == 2:
        d[:,1:-1] = 0.5 * (u[:,2:] - u[:,:-2]) / dth
        d[:,0] = 0.5 * u[:,1] * (1.0 - r_sym) / dth
        d[:,-1] = -0.5 * u[:,-1] * (1.0 - z_sym) / dth
    return d

In [ ]:
# One-dimensional versions.

def one_dim_Dr_u(u, order=4, r_sym=1):
    # Zero block like variable to differentiate.
    d = np.zeros_like(u)
    # Fourth order.
    if order == 4:
        # Centered stencil.
        d[2:-2] = (1.0 / 12.0) * (-u[4:] + 8.0 * u[3:-1] - 8.0 * u[1:-3] + u[:-4]) / dr
        # Parity ghost zones.
        d[1] = -r_sym * d[2]
        d[0] = -r_sym * d[3]
        # Boundaries use (-3,-2,-1,0,1) and (-4,-3,-2,-1,0) stencils.
        d[-2] = (1.0 / 12.0) * ( 3.0 * u[-1] + 10.0 * u[-2] - 18.0 * u[-3] +  6.0 * u[-4] -       u[-5]) / dr
        d[-1] = (1.0 / 12.0) * (25.0 * u[-1] - 48.0 * u[-2] + 36.0 * u[-3] - 16.0 * u[-4] + 3.0 * u[-5]) / dr
    # 2nd order.
    elif order == 2:
        d[1:-1] = 0.5 * (u[2:] - u[:-2]) / dr
        # Parity ghost zones.
        d[0] = -r_sym * d[1]
        # (-2,-1,0) stencil.
        d[-1] = 0.5 * (3.0 * u[-1] - 4.0 * u[-2] + u[-3]) / dr
    return d
    
# Same, but for z (second variable/index slot).
def one_dim_Dz_u(u, order=4, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[2:-2] = (1.0 / 12.0) * (-u[4:] + 8.0 * u[3:-1] - 8.0 * u[1:-3] + u[:-4]) / dz
        d[1] = -z_sym * d[2]
        d[0] = -z_sym * d[3]
        d[-2] = (1.0 / 12.0) * ( 3.0 * u[-1] + 10.0 * u[-2] - 18.0 * u[-3] +  6.0 * u[-4] -       u[-5]) / dz
        d[-1] = (1.0 / 12.0) * (25.0 * u[-1] - 48.0 * u[-2] + 36.0 * u[-3] - 16.0 * u[-4] + 3.0 * u[-5]) / dz
    elif order == 2:
        d[1:-1] = 0.5 * (u[2:] - u[:-2]) / dz
        d[0] = -z_sym * d[1]
        d[-1] = 0.5 * (3.0 * u[-1] - 4.0 * u[-2] + u[-3]) / dz
    return d

# Second derivative.
def one_dim_Drr_u(u, order=4, r_sym=1):
    d = np.zeros_like(u)
    # Fourth order.
    if order == 4:
        # Centered stencil.
        d[2:-2] = - (1.0 / 12.0) * (30.0 * u[2:-2] - 16.0 * (u[3:-1] + u[1:-3]) + (u[4:] + u[:-4])) / dr**2
        # Parity ghost zones.
        d[1] = r_sym * d[2]
        d[0] = r_sym * d[3]
        # Boundaries use (-4,-3,-2,-1,0,1) and (-5,-4,-3,-2,-1,0) stencils.
        d[-2] = (1.0 / 12.0) * (10.0 * u[-1] -  15.0 * u[-2] -   4.0 * u[-3] +  14.0 * u[-4] -  6.0 * u[-5] +        u[-6]) / dr**2
        d[-1] = (1.0 / 12.0) * (45.0 * u[-1] - 154.0 * u[-2] + 214.0 * u[-3] - 156.0 * u[-4] + 61.0 * u[-5] - 10.0 * u[-6]) / dr**2
    # 2nd order.
    elif order == 2:
        d[1:-1] = (u[2:] - 2.0 * u[1:-1] + u[:-2]) / dr**2
        # Parity ghost zones.
        d[0] = r_sym * d[1]
        # (-3,-2,-1,0) stencil.
        d[-1] = (2.0 * u[-1] - 5.0 * u[-2] + 4.0 * u[-3] - u[-4]) / dr**2
    return d
    
# Same, but for z.
def one_dim_Dzz_u(u, order=4, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[2:-2] = - (1.0 / 12.0) * (30.0 * u[2:-2] - 16.0 * (u[3:-1] + u[1:-3]) + (u[4:] + u[:-4])) / dz**2
        d[1] = z_sym * d[2]
        d[0] = z_sym * d[3]
        d[-2] = (1.0 / 12.0) * (10.0 * u[-1] -  15.0 * u[-2] -   4.0 * u[-3] +  14.0 * u[-4] -  6.0 * u[-5] +        u[-6]) / dz**2
        d[-1] = (1.0 / 12.0) * (45.0 * u[-1] - 154.0 * u[-2] + 214.0 * u[-3] - 156.0 * u[-4] + 61.0 * u[-5] - 10.0 * u[-6]) / dz**2
    elif order == 2:
        d[1:-1] = (u[2:] - 2.0 * u[1:-1] + u[:-2]) / dz**2
        d[0] = z_sym * d[1]
        d[-1] = (2.0 * u[:-1] - 5.0 * u[:-2] + 4.0 * u[:-3] - u[:-4]) / dz**2
    return d

def one_dim_Dzzz_u(u, order=4, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[3:-3] = (1.0 / 8.0) * (-13.0 * (u[4:-2] - u[2:-4]) + 8.0 * (u[5:-1] - u[1:-5]) - (u[6,:] - u[:,-6])) / dz**3
        d[2] = (1.0 / 8.0) * (-13.0 * (u[3] - u[1]) + 8.0 * (u[4] - u[0]) - (u[5] - z_sym * u[4])) / dz**3
        d[0] = -z_sym * d[3]
        d[1] = -z_sym * d[2]
        d[-3] = (1.0 / 8.0) * (       u[-1] +   8.0 * u[-2] -  35.0 * u[-3] +  48.0 * u[-4] -  29.0 * u[-5] +   8.0 * u[-6] -        u[-7]) / dz**3
        d[-2] = (1.0 / 8.0) * (15.0 * u[-1] -  56.0 * u[-2] +  83.0 * u[-3] -  64.0 * u[-4] +  29.0 * u[-5] -   8.0 * u[-6] +        u[-7]) / dz**3
        d[-1] = (1.0 / 8.0) * (49.0 * u[-1] - 232.0 * u[-2] + 461.0 * u[-3] - 496.0 * u[-4] + 307.0 * u[-5] - 104.0 * u[-6] + 15.0 * u[-7]) / dz**3
    elif order == 2:
        d[2:-2] = 0.5 * (-(u[:-4] - u[4:]) + 2.0 * (u[1:-3] - u[3:-1])) / dz**3
        d[1] = 0.5 * (-z_sym * u[2] + u[3] + 2.0 * u[0] - 2.0 * u[2]) / dz**3
        d[0] = -z_sym * d[1]
        d[-2] = 0.5 * (3.0 * u[-1] - 10.0 * u[-2] + 12.0 * u[-3] -  6.0 * u[-4] +       u[-5]) / dz**3
        d[-1] = 0.5 * (5.0 * u[-1] - 18.0 * u[-2] + 24.0 * u[-3] - 14.0 * u[-4] + 3.0 * u[-5]) / dz**3
    return d

Some relevant variables analysis.

In [ ]:
# Grab center interior points.
log_alpha = log_alpha_f
Dr_log_alpha = Dr_u(log_alpha)
Dz_log_alpha = Dz_u(log_alpha)

beta = beta_f
Dr_beta = Dr_u(beta)
Dz_beta = Dz_u(beta)

log_a = log_a_f
Dr_log_a = Dr_u(log_a)
Dz_log_a = Dz_u(log_a)

log_h = log_h_f
Dr_log_h = Dr_u(log_h)
Dz_log_h = Dz_u(log_h)

psi = psi_f
Dr_psi = Dr_u(psi)
Dz_psi = Dz_u(psi)

# Radial derivatives.
D_rr_log_alpha = (r / rr) * Dr_log_alpha + (z / rr) * Dz_log_alpha
D_rr_beta      = (r / rr) * Dr_beta + (z / rr) * Dz_beta
D_rr_log_a = (r / rr) * Dr_log_a + (z / rr) * Dz_log_a
D_rr_log_h = (r / rr) * Dr_log_h + (z / rr) * Dz_log_h
D_rr_psi   = (r / rr) * Dr_psi + (z / rr) * Dz_psi

Radial metric coefficient.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(rr[:,ghost], np.exp(2.0 * log_a[:,ghost]), label = r"$\rho\,:\,A^2$")
ax.plot(rr[ghost,:], np.exp(2.0 * log_a[ghost,:]), label = r"$z\,:\,A^2$")
ax.plot(rr[ghost,:], np.ones_like(rr[ghost,:]), label = r"$1$")
ax.set_xlabel(r"$r$")
#ax.set_ylim(-0.01,0.01)
#ax.xaxis.set_major_locator(MultipleLocator(2))
#ax.xaxis.set_minor_locator(MultipleLocator(0.25))
#ax.yaxis.set_major_locator(MultipleLocator(0.01))
#ax.yaxis.set_minor_locator(MultipleLocator(0.005))
ax.legend()
ax.set_title(r"$g_{rr} = A^2$")
plt.show()

Angular metric coefficient.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(rr[:,ghost], np.exp(2.0 * log_a[:,ghost]), label = r"$\rho\,:\,H^2$")
ax.plot(rr[ghost,:], np.exp(2.0 * log_a[ghost,:]), label = r"$z\,:\,H^2$")
ax.plot(rr[ghost,:], np.ones_like(rr[ghost,:]), label = r"$1$")
ax.set_xlabel(r"$r$")
#ax.set_ylim(-0.01,0.01)
#ax.xaxis.set_major_locator(MultipleLocator(2))
#ax.xaxis.set_minor_locator(MultipleLocator(0.25))
#ax.yaxis.set_major_locator(MultipleLocator(0.01))
#ax.yaxis.set_minor_locator(MultipleLocator(0.005))
ax.legend()
ax.set_title(r"$g_{\varphi \varphi}/ \rho^2 = H^2$")
plt.show()

As we go to spatial infinity, we should get $A = H$.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(rr[:,ghost], np.exp(2.0 * (log_a[:,ghost] - log_h[:,ghost])), label = r"$\rho\,:\,A^2/H^2$")
ax.plot(rr[ghost,:], np.exp(2.0 * (log_a[ghost,:] - log_h[ghost,:])), label = r"$z\,:\,A^2/H^2$")
ax.plot(rr[ghost,:], np.ones_like(rr[ghost,:]), label = r"$1$")
ax.set_xlabel(r"$r$")
#ax.set_ylim(-0.01,0.01)
#ax.xaxis.set_major_locator(MultipleLocator(2))
#ax.xaxis.set_minor_locator(MultipleLocator(0.25))
#ax.yaxis.set_major_locator(MultipleLocator(0.01))
#ax.yaxis.set_minor_locator(MultipleLocator(0.005))
ax.legend()
ax.set_title(r"$\rho^2 g_{rr} / g_{\varphi \varphi} = A^2/H^2$")
plt.show()

We will first examine how the variables $\alpha,\,A,\,H$ decay. These variables should decay as

$$\alpha = 1 - \frac{M}{r}\,,\quad H = A = 1 + \frac{M}{r}\,.$$

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(rr[:,ghost], rr[:,ghost] * -(np.exp(log_alpha[:,ghost]) - 1.0), label = r"$r (1 - \alpha)\,:\,\rho$")
ax.plot(rr[ghost,:], rr[ghost,:] * -(np.exp(log_alpha[ghost,:]) - 1.0), label = r"$r (1 - \alpha)\,:\,z$")

ax.plot(rr[:,ghost], rr[:,ghost] * (np.exp(log_a[:,ghost]) - 1.0), label = r"$r (A - 1)\,:\,\rho$")
ax.plot(rr[ghost,:], rr[ghost,:] * (np.exp(log_a[ghost,:]) - 1.0), label = r"$r (A - 1)\,:\,z$")

ax.plot(rr[:,ghost], rr[:,ghost] * (np.exp(log_h[:,ghost]) - 1.0), label = r"$r (H - 1)\,:\,\rho$")
ax.plot(rr[ghost,:], rr[ghost,:] * (np.exp(log_h[ghost,:]) - 1.0), label = r"$r (H - 1)\,:\,z$")

ax.plot(rr[:,ghost], np.sqrt(-rr[:,ghost]**4 * (lambda_f[:,ghost])), label = r"$r^4 \lambda\,:\,\rho$")
ax.plot(rr[ghost,:], np.sqrt(-rr[ghost,:]**4 * (lambda_f[ghost,:])), label = r"$r^4 \lambda\,:\,z$")

ax.set_xlabel(r"$r$")
#ax.set_xlim(32)
#ax.set_ylim(1.95,2.1)
#ax.xaxis.set_major_locator(MultipleLocator(2))
#ax.xaxis.set_minor_locator(MultipleLocator(0.25))
#ax.yaxis.set_major_locator(MultipleLocator(0.01))
#ax.yaxis.set_minor_locator(MultipleLocator(0.005))
ax.legend()
ax.set_title(r"Robin Decay")
plt.show()

## Spherical Interpolation

In [ ]:
sph_rr = np.genfromtxt(dir_name + "sph_rr.asc")
sph_th = np.genfromtxt(dir_name + "sph_th.asc")

In [ ]:
drr = sph_rr[1,0] - sph_rr[0,0]
dth = sph_th[0,1] - sph_th[0,0]

In [ ]:
sph_log_alpha = np.genfromtxt(dir_name + "sph_log_alpha_f.asc")
sph_beta  = np.genfromtxt(dir_name + "sph_beta_f.asc")
sph_log_a = np.genfromtxt(dir_name + "sph_log_a_f.asc")
sph_log_h = np.genfromtxt(dir_name + "sph_log_h_f.asc")
sph_psi   = np.genfromtxt(dir_name + "sph_psi_f.asc")

In [ ]:
X, Y = sph_rr * np.sin(sph_th), sph_rr * np.cos(sph_th)

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, np.exp(sph_log_alpha), cmap=plt.cm.viridis)
#ax.set_zlim(0,1)
ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$z$")
ax.set_zlabel(r"$\alpha$")
fig.colorbar(surf, shrink=0.5, aspect=5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(r[:,ghost], np.exp(log_alpha[:,ghost]), label = r"$\alpha(\rho,0)$")
ax.plot(z[ghost,:], np.exp(log_alpha[ghost,:]), label = r"$\alpha(0,z)$")
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\alpha$")
ax.legend()
plt.show()

## The Schwarzschild Pseudomass.

We can calculate the Schwarzschild pseudomass by first obtaining the area of 2-spheres.

$$M = \left(\frac{\tilde{A}}{16\pi}\right)^{1/2}\,\left(1 - \frac{(d\tilde{A}/dr)^2}{16\pi \tilde{g}_{rr} \tilde{A}}\right)\,,$$

where $\tilde{g}_{rr}$ is the average of $g_{rr}$ over the same 2-sphere, and $\tilde{A}$ is the 2-sphere area.

Recall that our metric has the following form in spherical coordinates:

$$dl^2 = A^2\,dr^2 + r^2\,\left(A^2\,d\theta^2 + H^2\,\sin^2\theta\,d\varphi^2\right)\,.$$

Metric on 2-spheres is 

$$\Omega_{ab} = r^2\,\left(A^2\,d\theta^2 + H^2\,\sin^2\theta^2\,d\varphi^2\right)\,,$$

Thus,

$$\tilde{A} = \int_S\,\sqrt{\Omega}\,d\theta\,d\varphi = 2\pi\,r^2\,\int\limits_0^\pi\,AH\,\sin\theta\,d\theta\,.$$

Simillarly, 

$$\tilde{g}_{rr} = \frac{1}{\tilde{A}}\,\int_S\,\,A^2\,\sqrt{\Omega}\,d\theta\,d\varphi = \frac{2\pi\,r^2}{\tilde{A}}\,\int\limits_0^\pi\,A^3H\,\sin\theta\,d\theta\,.$$


By approximating that $A$ and $H$ do not have angular dependence far away, we can say that 

$$\tilde{A} = 4\pi\,r^2\,AH\,.$$

Thus, 

$$\frac{d \tilde{A}}{dr} = 8\pi\,AH\,r\,\left(1 + \frac{r}{2}\,\left(\partial_r \log A + \partial_r \log H\right)\right)\,.$$

$$M = \frac{r\,\sqrt{AH}}{2}\,\left(1 - \frac{H}{A}\,\left(1 + \frac{r}{2}\,\left(\partial_r \log A + \partial_r \log H \right)^2\right)\right)$$

In [ ]:
# Calculate cartesian expression assuming no angular dependence.
car_M_S = 0.5 * rr * np.exp(0.5 * (log_a + log_h)) * (1.0 - np.exp(log_h - log_a) * (1.0 + 0.5 * rr * (D_rr_log_a + D_rr_log_h))**2)

In [ ]:
sph_Drr_log_a = two_dim_Drr_u(sph_log_a)
sph_Drr_log_h = two_dim_Drr_u(sph_log_h)
# Now calculate true expression with angular integration.
i0 = integrate.simps(np.exp(sph_log_a + sph_log_h) * np.sin(sph_th), sph_th[0,:], axis=-1)
i1 = integrate.simps(np.exp(3.0 * sph_log_a + sph_log_h) * np.sin(sph_th), sph_th[0,:], axis=-1)
i2 = integrate.simps(np.exp(sph_log_a + sph_log_h) * np.sin(sph_th) * (1.0 + 0.5 * (sph_rr * sph_Drr_log_a + sph_rr * sph_Drr_log_h)), sph_th[0,:], axis=-1)
# Schwarzschild Pseudomass
sph_M_S = 0.5 * sph_rr[:,0] * np.sqrt(i0) * (1.0 - i2**2 / i1)

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

# Cartesian expressions with no angular integration.
ax.plot((rr[:,ghost])[rr[:,ghost] > 0.0], (car_M_S[:,ghost])[rr[:,ghost] > 0.0], label = r"Approx. $M_{S}(\rho)$")
ax.plot((rr[ghost,:])[rr[ghost,:] > 0.0], (car_M_S[ghost,:])[rr[ghost,:] > 0.0], label = r"Approx. $M_{S}(z)$")
ax.plot((np.diagonal(rr))[np.diagonal(rr) > 0.0], (np.diagonal(car_M_S))[np.diagonal(rr) > 0.0], label = r"Approx. $M_{S}(d)$")

# True expression.
ax.plot(sph_rr[:,0], sph_M_S, label = r"True $M_S(r)$")

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$M_{S}$")
ax.legend()
ax.set_title(r"Schwarzschild Pseudo-Mass $M_S$")
plt.show()

Report last value and standard deviation of last 100 points.

In [ ]:
sph_M_S[-1], np.std(sph_M_S[-100:])

## Komar Mass

We use Gourgoulhon's eq. (4.14) and (4.15) in Gourgoulhon (2011) for the Komar mass.

\begin{align}
M_{\text{Komar}} =& \frac{1}{4\pi}\,\lim_{\mathscr{S}\to \infty}\,\oint_\mathscr{S}\,\left(\frac{\partial \alpha}{\partial r} - \frac{H^2\,r^2\sin^2\theta}{2\alpha}\,\Omega\,\frac{\partial \Omega}{\partial r}\right)\,H\,r^2\,\sin\theta\,d\theta\,d\varphi\,,\\
M_{\text{Komar}} =& \frac{1}{4\pi}\,\lim_{\mathscr{S}\to \infty}\,\oint_\mathscr{S}\,\frac{\partial \alpha}{\partial r}\,r^2\,\sin\theta\,d\theta\,d\varphi\,,\\
M_{\text{Komar}} =&\,\int_{\Sigma_t}\,\left(\frac{2\omega}{\alpha}\,(\omega + l \Omega) - \alpha\,m^2 \right)\,\phi^2\,\,A^2\,H\,r^2\,\sin\theta\,dr\,d\theta\,d\varphi\,.
\end{align}

In [ ]:
sph_Drr_alpha = np.exp(sph_log_alpha) * two_dim_Drr_u(sph_log_alpha)
sph_Drr_beta = two_dim_Drr_u(sph_beta)
sph_phi = (sph_rr * np.sin(sph_th))**l * sph_psi
sph_M_K1 = integrate.simps((sph_Drr_alpha - 0.5 * np.exp(2.0 * sph_log_h) * (sph_rr * np.sin(sph_th))**2 * sph_beta * sph_Drr_beta / np.exp(sph_log_alpha)) * np.exp(sph_log_h) * sph_rr**2 * np.sin(sph_th), sph_th[0,:], axis=-1)
sph_M_K2 = integrate.simps(sph_Drr_alpha * sph_rr**2 * np.sin(sph_th), sph_th[0,:], axis=-1)
i_M_K3 = 4.0 * np.pi * (2.0 * w * (w + l * sph_beta) / np.exp(sph_log_alpha) - np.exp(sph_log_alpha) * m**2) * sph_phi**2 * np.exp(2.0 * sph_log_a + sph_log_h) * sph_rr**2 * np.sin(sph_th)
sph_M_K3 = np.zeros_like(sph_rr[:,0])
for i in range(sph_M_K3.size):
    if (i > 1):
        sph_M_K3[i] = integrate.simps(integrate.simps(i_M_K3[:i,:], sph_th[0,:], axis=-1), sph_rr[:i,0], even='avg')

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

# True expression.
ax.plot(sph_rr[:,0], sph_M_K1, label = r"$M_{K1}(r)$")
ax.plot(sph_rr[:,0], sph_M_K2, label = r"$M_{K2}(r)$")
ax.plot(sph_rr[:,0], sph_M_K3, label = r"$M_{K3}(r)$")

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$M_K$")
ax.legend()
ax.set_title(r"Komar Masses Comparison")
plt.show()

## ADM Mass.

We can use Gourgoulhon's expression, see eq. (4.20) in Gourgoulhon (2011).

$$M_{\text{ADM}} = -\frac{1}{16\pi}\,\lim_{\mathscr{S}\to\infty}\,\oint_{\mathscr{S}}\,\left[ \frac{\partial}{\partial r}\,(A^2 + H^2) + \frac{H^2 - A^2}{r}\right]\,r^2 \sin\theta\,d\theta\,d\varphi$$

By once again approximating that there is no angular dependence, we reach the expression

$$M_{\text{ADM}} = -\frac{1}{4}\,\left(2r^2\left(A^2\,\partial_r \log A + H^2\,\partial_r \log H\right) + r\,(H^2 - A^2)\right)$$

In [ ]:
sph_Drr_log_a = two_dim_Drr_u(sph_log_a)
sph_Drr_log_h = two_dim_Drr_u(sph_log_h)

In [ ]:
sph_M_ADM = -0.25 * integrate.simps(sph_rr * np.sin(sph_th) * (np.exp(2.0 * sph_log_a) * (2.0 * sph_rr * sph_Drr_log_a - 1.0) + np.exp(2.0 * sph_log_h) * (2.0 * sph_rr * sph_Drr_log_h + 1.0)), sph_th[0,:], axis = -1)

In [ ]:
car_M_ADM = -0.25 * (2.0 * rr**2 * (np.exp(2.0 * log_a) * D_rr_log_a + np.exp(2.0 * log_h) * D_rr_log_h) + rr * (np.exp(2.0 * log_h) - np.exp(2.0 * log_a)))

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

# Cartesian expressions with no angular integration.
ax.plot((rr[:,ghost])[rr[:,ghost] > 0.0], (car_M_ADM[:,ghost])[rr[:,ghost] > 0.0], label = r"Approx. $M_{ADM}(\rho)$")
ax.plot((rr[ghost,:])[rr[ghost,:] > 0.0], (car_M_ADM[ghost,:])[rr[ghost,:] > 0.0], label = r"Approx. $M_{ADM}(z)$")
ax.plot((np.diagonal(rr))[np.diagonal(rr) > 0.0], (np.diagonal(car_M_ADM))[np.diagonal(rr) > 0.0], label = r"Approx. $M_{ADM}(d)$")

# True expression.
ax.plot(sph_rr[:,0], sph_M_ADM, label = r"True $M_{ADM}(r)$")

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$M_{ADM}$")
ax.legend()
ax.set_title(r"ADM Mass $M_{ADM}$")
plt.show()

In [ ]:
sph_M_ADM[-1], np.std(sph_M_ADM[sph_rr[:,0] > sph_rr[-1,0] - 2.0])

# Angular Momentum.

We can use Gourgoulhon's expression, see eq. (4.37) in Gourgoulhon (2011).

\begin{align}
J =&\, \frac{1}{16\pi}\,\lim_{\mathscr{S}\to\infty}\,\oint_{\mathscr{S}}\,\frac{H^3\,r^4\,\sin^3\theta}{\alpha}\,\frac{\partial \Omega}{\partial r}\,d\theta\,d\varphi\,,\\
J =&\, \frac{1}{16\pi}\,\lim_{\mathscr{S}\to\infty}\,\oint_{\mathscr{S}}\,r^4\,\sin^3\theta\,\frac{\partial \Omega}{\partial r}\,d\theta\,d\varphi\,,\\
J =&\, l\,\int_{\Sigma_t}\,\frac{(\omega + l\Omega)}{\alpha}\,\phi^2\,A^2\,H\,r^2\,\sin\theta\,dr\,d\theta\,d\varphi\,.
\end{align}

In [ ]:
sph_Drr_beta = two_dim_Drr_u(sph_beta)
i_J1 = 0.25 * np.exp(3.0 * sph_log_h) * sph_rr**4 * np.sin(sph_th)**3 * sph_Drr_beta / np.exp(sph_log_alpha)
sph_J1 = integrate.simps(i_J1, sph_th[0,:], axis=-1)
i_J2 = 0.25 * sph_rr**4 * np.sin(sph_th)**3 * sph_Drr_beta
sph_J2 = integrate.simps(i_J2, sph_th[0,:], axis=-1)

In [ ]:
i_J3 = 4.0 * np.pi * l * (w + l * sph_beta) * sph_phi**2 * np.exp(2.0 * sph_log_a + sph_log_h) * sph_rr**2 * np.sin(sph_th) / np.exp(sph_log_alpha)
sph_J3 = np.zeros_like(sph_rr[:,0])
for i in range(sph_J3.size):
    if (i > 1):
        sph_J3[i] = integrate.simps(integrate.simps(i_J3[:i,:], sph_th[0,:], axis=-1), sph_rr[:i,0], even='avg')

In [ ]:
car_J = D_rr_beta * rr**4 / 6.0

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(sph_rr[:,0], sph_J1, label = r"$J_{1}(r)$")
ax.plot(sph_rr[:,0], sph_J2, label = r"$J_{2}(r)$")
ax.plot(sph_rr[:,0], sph_J3, label = r"$J_{3}(r)$")

# Cartesian expressions with no angular integration.
ax.plot((rr[:,ghost])[rr[:,ghost] > 0.0], (car_J[:,ghost])[rr[:,ghost] > 0.0], label = r"Approx. $J(\rho)$")
ax.plot((rr[ghost,:])[rr[ghost,:] > 0.0], (car_J[ghost,:])[rr[ghost,:] > 0.0], label = r"Approx. $J(z)$")
ax.plot((np.diagonal(rr))[np.diagonal(rr) > 0.0], (np.diagonal(car_J))[np.diagonal(rr) > 0.0], label = r"Approx. $J(d)$")

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$J$")
ax.legend()
ax.set_title(r"Angular Momentum Comparison")
plt.show()

## Calculate $a$ and $M$ from Kerr metric.

$$a = - \frac{r^2 H^2 \Omega}{1 - \alpha^2 + \rho^2 H^2 \Omega^2}\,.$$

$$M^3 + 4R\, M^2 + (4R^2 - a^2)\,M - 2R^3 A^2\,\left(1 - \alpha^2 + \rho^2 H^2 \Omega^2\right) = 0\,.$$

In [ ]:
i_v = 4.0 * np.pi * (np.exp(sph_log_a + sph_log_h) * np.sin(sph_th))
i_a = i_v * (-sph_rr**2 * np.exp(2.0 * sph_log_h) * sph_beta / (1.0 - np.exp(2.0 * sph_log_alpha) + np.exp(2.0 * sph_log_h) * (sph_rr * np.sin(sph_th) * sph_beta)**2))
sph_a = integrate.simps(i_a, sph_th[0,:], axis=-1) / integrate.simps(i_v, sph_th[0,:], axis=-1)

In [ ]:
cart_a = - rr**2 * beta * np.exp(2.0 * log_h) / ((1.0 - np.exp(2.0 * log_alpha))  + np.exp(2.0 * log_h) * beta**2 * r**2)

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

#ax.plot((rr[:,ghost])[rr[:,ghost] < 125.0], (cart_a[:,ghost])[rr[:,ghost] < 125.0], label = r"$a(\rho)$")
#ax.plot((rr[ghost,:])[rr[ghost,:] < 125.0], (cart_a[ghost,:])[rr[ghost,:] < 125.0], label = r"$a(z)$")

ax.plot((np.diagonal(rr))[np.diagonal(rr) < 125.0], (np.diagonal(cart_a))[np.diagonal(rr) < 125.0], label = r"Approx. $a(d)$")

ax.plot(sph_rr[:,0], sph_a, label = r"True $a(r)$")

#ax.set_xlim(32)
#ax.set_ylim(2.0,2.25)

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$a$")
ax.legend()
ax.set_title(r"Kerr Angular Momentum Ratio $a=J/M$")
plt.show()

In [ ]:
c1 = np.ones_like(rr).astype(complex)
c2 = (4.0 * rr).astype(complex)
c3 = (4.0 * rr**2 - cart_a**2).astype(complex)
c4 = (-2.0 * rr**3 * np.exp(2.0 * log_a) * (1.0 - np.exp(2.0 * log_alpha) + beta**2 * np.exp(2.0 * log_h) * r**2))

d0 = c2**2 - 3.0 * c1 * c3
d1 = 2.0 * c2**3 - 9.0 * c1 * c2 * c3 + 27.0 * c1**2 * c4

e0 = (0.5 * (d1 + (d1**2 - 4.0 * d0**3)**(0.5)))**(1.0 / 3.0)
f0 = 0.5 * (-1.0 + np.sqrt(3.0) * np.sqrt(-1.0 + 0.0j))

#M0 = -1.0 / (3.0 * c1) * (c2 + f0**0 * e0 + d0 / (f0**0 * e0))
M1 = -1.0 / (3.0 * c1) * (c2 + f0**1 * e0 + d0 / (f0**1 * e0))
#M2 = -1.0 / (3.0 * c1) * (c2 + f0**2 * e0 + d0 / (f0**2 * e0))

cart_M_Kerr = np.real(M1)

In [ ]:
l_rr = sph_rr[:,0]

c1 = np.ones_like(l_rr).astype(complex)
c2 = (4.0 * l_rr).astype(complex)
c3 = (4.0 * l_rr**2 - sph_a**2).astype(complex)
c4 = (-2.0 * l_rr**3 * np.exp(2.0 * sph_log_a[:,0]) * (1.0 - np.exp(2.0 * sph_log_alpha[:,0]) + sph_beta[:,0]**2 * np.exp(2.0 * sph_log_h[:,0]) * (l_rr * np.sin(sph_th[:,0]))**2))

d0 = c2**2 - 3.0 * c1 * c3
d1 = 2.0 * c2**3 - 9.0 * c1 * c2 * c3 + 27.0 * c1**2 * c4

e0 = (0.5 * (d1 + (d1**2 - 4.0 * d0**3)**(0.5)))**(1.0 / 3.0)
f0 = 0.5 * (-1.0 + np.sqrt(3.0) * np.sqrt(-1.0 + 0.0j))

#M0 = -1.0 / (3.0 * c1) * (c2 + f0**0 * e0 + d0 / (f0**0 * e0))
M1 = -1.0 / (3.0 * c1) * (c2 + f0**1 * e0 + d0 / (f0**1 * e0))
#M2 = -1.0 / (3.0 * c1) * (c2 + f0**2 * e0 + d0 / (f0**2 * e0))

sph_M_Kerr = np.real(M1)

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

#ax.plot((rr[:,ghost])[rr[:,ghost] < 125.0], (cart_M_Kerr[:,ghost])[rr[:,ghost] < 125.0], label = r"$a(\rho)$")
#ax.plot((rr[ghost,:])[rr[ghost,:] < 125.0], (cart_M_Kerr[ghost,:])[rr[ghost,:] < 125.0], label = r"$a(z)$")

ax.plot((np.diagonal(rr))[np.diagonal(rr) < 125.0], (np.diagonal(cart_M_Kerr))[np.diagonal(rr) < 125.0], label = r"Approx. $M_{Kerr}(d)$")

ax.plot(sph_rr[:,0], sph_M_Kerr, label = r"True $M_{Kerr}(r)$")

#ax.set_xlim(32)
#ax.set_ylim(2.0,2.25)

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$M_{Kerr}$")
ax.legend()
ax.set_title(r"Kerr Mass $M_{Kerr}$")
plt.show()

In [ ]:
sph_J_Kerr = sph_a * sph_M_Kerr
cart_J_Kerr = cart_a * cart_M_Kerr

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

#ax.plot((rr[:,ghost])[rr[:,ghost] < 125.0], (cart_J_Kerr[:,ghost])[rr[:,ghost] < 125.0], label = r"$a(\rho)$")
#ax.plot((rr[ghost,:])[rr[ghost,:] < 125.0], (cart_J_Kerr[ghost,:])[rr[ghost,:] < 125.0], label = r"$a(z)$")

ax.plot((np.diagonal(rr))[np.diagonal(rr) < 125.0], (np.diagonal(cart_J_Kerr))[np.diagonal(rr) < 125.0], label = r"Approx. $J_{Kerr}(d)$")

ax.plot(sph_rr[:,0], sph_J_Kerr, label = r"True $J_{Kerr}(r)$")

#ax.set_xlim(32)
#ax.set_ylim(2.0,2.25)

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$J_{Kerr}$")
ax.legend()
ax.set_title(r"Kerr Angular Momentum $J_{Kerr}$")
plt.show()

# Full Comparison

In [ ]:
plt.rcParams.update({'font.size': 12})

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(sph_rr[:,0], sph_M_S, label = r"$M_{PS}$")
ax.plot(sph_rr[:,0], sph_M_K1, label = r"S($M_{Komar})$")
ax.plot(sph_rr[:,0], sph_M_K3, label = r"V($M_{Komar})$")
ax.plot(sph_rr[:,0], sph_M_ADM, label = r"$M_{ADM}$")

ax.set_xlabel(r"$r\,[\hbar/m]$")
ax.set_ylabel(r"$M\,[m_P^2/m]$")
ax.legend()
ax.set_title(r"Masses Comparison for $l = %d$, $\omega = %.3f\,(m/\hbar)$" % (l, w_f))
plt.show()

Report average from $M_{Schwarzschild},\, M_{Komar_1},\, M_{Komar_3},\,M_{Kerr}$ with standard deviation.

In [ ]:
np.average(np.array([sph_M_S[-1], sph_M_K1[-1], sph_M_K3[-1], sph_M_Kerr[-1]])), np.std(np.array([sph_M_S[-1], sph_M_K1[-1], sph_M_K3[-1], sph_M_Kerr[-1]]))

The "best" expressions are the Komar masses, $M_{Komar_1}$ and $M_{Komar_3}$.

In [ ]:
M, re_M, e_M = 0.5 * (sph_M_K1[-1] + sph_M_K3[-1]), 2.0 * np.abs(sph_M_K1[-1] - sph_M_K3[-1]) / np.abs(sph_M_K1[-1] + sph_M_K3[-1]), np.abs(sph_M_K1[-1] - sph_M_K3[-1])

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

#ax.plot(sph_rr[:,0], sph_J_S, label = r"$M_{PS}(r)$")
ax.plot(sph_rr[:,0], sph_J1, label = r"S($J_{Komar})$")
ax.plot(sph_rr[:,0], sph_J3, label = r"V($J_{Komar})$")
ax.plot(sph_rr[:,0], sph_J2, label = r"$J_{Asymptotic}$")
#ax.plot(sph_rr[:,0], sph_J_ADM, label = r"$M_{ADM}(r)$")

ax.set_xlabel(r"$r\,[\hbar/m]$")
ax.set_ylabel(r"$J\,[(m_P^2/m)^2]$")
ax.legend()
ax.set_title(r"Angular Momenta Comparison for $l = %d$, $\omega = %.3f\,(m/\hbar)$" % (l, w_f))
plt.show()

Report $J_{Komar_1},\,J_{Komar_3}$:

In [ ]:
0.5 * (sph_J1[-1] + sph_J3[-1]), 2.0 * np.abs(sph_J1[-1] - sph_J3[-1]) / np.abs(sph_J1[-1] + sph_J3[-1]), np.abs(sph_J1[-1] - sph_J3[-1])

# Error Indicators

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(rr[:,ghost], rr[:,ghost] * -(np.exp(log_alpha[:,ghost]) - 1.0), label = r"$r (1 - \alpha)\,:\,\rho$")
ax.plot(rr[ghost,:], rr[ghost,:] * -(np.exp(log_alpha[ghost,:]) - 1.0), label = r"$r (1 - \alpha)\,:\,z$")

ax.plot(rr[:,ghost], rr[:,ghost] * (np.exp(log_a[:,ghost]) - 1.0), label = r"$r (A - 1)\,:\,\rho$")
ax.plot(rr[ghost,:], rr[ghost,:] * (np.exp(log_a[ghost,:]) - 1.0), label = r"$r (A - 1)\,:\,z$")

ax.plot(rr[:,ghost], rr[:,ghost] * (np.exp(log_h[:,ghost]) - 1.0), label = r"$r (H - 1)\,:\,\rho$")
ax.plot(rr[ghost,:], rr[ghost,:] * (np.exp(log_h[ghost,:]) - 1.0), label = r"$r (H - 1)\,:\,z$")

ax.plot(rr[ghost,:], M * np.ones_like(rr[ghost,:]), "--", label = r"$M$")

ax.set_xlabel(r"$r$")
#ax.set_xlim(0,32)
ax.set_ylim(0.95 * M, 1.05 * M)
#ax.xaxis.set_major_locator(MultipleLocator(2))
#ax.xaxis.set_minor_locator(MultipleLocator(0.25))
#ax.yaxis.set_major_locator(MultipleLocator(0.01))
#ax.yaxis.set_minor_locator(MultipleLocator(0.005))
ax.legend()
ax.set_title(r"Robin Decay")
plt.show()

In [ ]:
np.abs(sph_M_K1[-1] - sph_M_K3[-1]), np.abs(sph_J1[-1] - sph_J3[-1])

# Apparent Horizon

In [ ]:
# Make an array from which to shoot from.
initial_h = sph_rr[:,0][sph_rr[:,0] < 4 * M][:1:-1]

In [ ]:
h = np.zeros((initial_h.size, sph_th[0,:].size))
dh = np.zeros((initial_h.size, sph_th[0,:].size))

In [ ]:
print(initial_h.size)

In [ ]:
# Will need variables log_h and log_a and its derivatives.
sph_Dth_log_a = two_dim_Dth_u(sph_log_a)
sph_Dth_log_h = two_dim_Dth_u(sph_log_h)

In [ ]:
# Interpolation functions.
i_log_a = interpolate.RectBivariateSpline(sph_rr[:,0], sph_th[0,:], sph_log_a)
i_Drr_log_a = interpolate.RectBivariateSpline(sph_rr[:,0], sph_th[0,:], sph_Drr_log_a)
i_Dth_log_a = interpolate.RectBivariateSpline(sph_rr[:,0], sph_th[0,:], sph_Dth_log_a)

i_log_h = interpolate.RectBivariateSpline(sph_rr[:,0], sph_th[0,:], sph_log_h)
i_Drr_log_h = interpolate.RectBivariateSpline(sph_rr[:,0], sph_th[0,:], sph_Drr_log_h)
i_Dth_log_h = interpolate.RectBivariateSpline(sph_rr[:,0], sph_th[0,:], sph_Dth_log_h)

In [ ]:
def source_h(th, h, dh, log_a, Drr_log_a, Dth_log_a, log_h, Drr_log_h, Dth_log_h):
    s_h = dh
    if (th == 0.0):
        s_dh = h * (0.5 * h * (Drr_log_a + Drr_log_h) + 1.0)
    else:
        s_dh = (1.0 + (dh / h)**2) * (h**2 * (Drr_log_a + Drr_log_h) - dh * (Dth_log_a + Dth_log_h)) + 2.0 * h + 3.0 * dh**2 / h - cos(th) * (1.0  + (dh / h)**2) * (dh / sin(th))
    return np.array([s_h, s_dh])

In [ ]:
for i in range(initial_h.size):
    h[i,0] = h0 = initial_h[i]
    dh[i,0] = dh0 = 0.0
    # Integrate using RK4.
    for j in range(sph_th[0,:].size - 1):
        i_t = sph_th[0,j]
        i_y0 = h[i,j]
        i_y1 = dh[i,j]
        k1 = dth * source_h(i_t, i_y0, i_y1,
                            i_log_a(i_y0, i_t), i_Drr_log_a(i_y0, i_t), i_Dth_log_a(i_y0, i_t),
                            i_log_h(i_y0, i_t), i_Drr_log_h(i_y0, i_t), i_Dth_log_h(i_y0, i_t))
        k2 = dth * source_h(i_t + 0.5 * dth, i_y0 + 0.5 * k1[0], i_y1 + 0.5 * k1[1],
                            i_log_a(i_y0 + 0.5 * k1[0], i_t + 0.5 * dth), i_Drr_log_a(i_y0 + 0.5 * k1[0], i_t + 0.5 * dth), i_Dth_log_a(i_y0 + 0.5 * k1[0], i_t + 0.5 * dth),
                            i_log_h(i_y0 + 0.5 * k1[0], i_t + 0.5 * dth), i_Drr_log_h(i_y0 + 0.5 * k1[0], i_t + 0.5 * dth), i_Dth_log_h(i_y0 + 0.5 * k1[0], i_t + 0.5 * dth))
        k3 = dth * source_h(i_t + 0.5 * dth, i_y0 + 0.5 * k2[0], i_y1 + 0.5 * k2[1],
                            i_log_a(i_y0 + 0.5 * k2[0], i_t + 0.5 * dth), i_Drr_log_a(i_y0 + 0.5 * k2[0], i_t + 0.5 * dth), i_Dth_log_a(i_y0 + 0.5 * k2[0], i_t + 0.5 * dth),
                            i_log_h(i_y0 + 0.5 * k2[0], i_t + 0.5 * dth), i_Drr_log_h(i_y0 + 0.5 * k2[0], i_t + 0.5 * dth), i_Dth_log_h(i_y0 + 0.5 * k2[0], i_t + 0.5 * dth))
        k4 = dth * source_h(i_t + dth, i_y0 + k3[0], i_y1 + k3[1],
                            i_log_a(i_y0 + k3[0], i_t + dth), i_Drr_log_a(i_y0 + k3[0], i_t + dth), i_Dth_log_a(i_y0 + k3[0], i_t + dth),
                            i_log_h(i_y0 + k3[0], i_t + dth), i_Drr_log_h(i_y0 + k3[0], i_t + dth), i_Dth_log_h(i_y0 + k3[0], i_t + dth))
        h[i,j+1] = i_y0 + (k1[0] + 2.0 * k2[0] + 2.0 * k3[0] + k4[0]) / 6.0
        dh[i,j+1] = i_y1 + (k1[1] + 2.0 * k2[1] + 2.0 * k3[1] + k4[1]) / 6.0

In [ ]:
# Check if derivative crossed zero.
np.any(dh[:,-1] < 0.0)

In [ ]:
# Print derivative minimum and rr value.
np.min(dh[:-1,-1]), initial_h[np.argmin(dh[:-1,-1])]

In [ ]:
fig, ax = plt.subplots(figsize = (8,6))
ax.plot(initial_h[:], np.log10(np.abs(dh[:,-1])))
#ax.plot(initial_h[:], dh[:,-1])
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\log_{10}|dh/d\theta|$")
ax.set_title(r"Apparent Horizon Condition $dh/d\theta\,(\theta = \pi/2)$")
plt.show()

# Axis Behavior

In [ ]:
fig, ax = plt.subplots(figsize = (7, 6))
ax.plot(r[:16,2], ((np.exp(2.0 * log_a_f) - np.exp(2.0 * log_h_f))/(r/dr)**2)[:16,2], ".-")

ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$\lambda$")
ax.set_title(r"Regularization $\lambda_f$ profile on equator")

plt.show()

In [ ]:
alpha = np.exp(log_alpha_f)
H = np.exp(2.0 * log_h_f)
A = np.exp(2.0 * log_a_f)
psi = psi_f

In [ ]:
Dr_alpha = Dr_u(alpha)
Dz_alpha = Dz_u(alpha)

Dr_H = Dr_u(H)
Dz_H = Dz_u(H)

Dr_A = Dr_u(A)
Dz_A = Dz_u(A)

Dr_psi = Dr_u(psi)
Dz_psi = Dz_u(psi)

In [ ]:
Drr_alpha = Drr_u(alpha)
Drr_H = Drr_u(H)
Drr_A = Drr_u(A)
Drr_psi = Drr_u(psi)

In [ ]:
def axis_i(u, Dr_u, Drr_u):
    return u[ghost,:] - (5.0 / 16.0) * dr * Dr_u[ghost,:] + (1.0 / 32.0) * dr**2 * Drr_u[ghost,:]

In [ ]:
axis_alpha = axis_i(alpha , Dr_alpha, Drr_alpha)
axis_H = axis_i(H , Dr_H, Drr_H)
axis_A = axis_i(A , Dr_A, Drr_A)
axis_psi = axis_i(psi , Dr_psi, Drr_psi)

In [ ]:
axis_Dz_alpha = one_dim_Dz_u(axis_alpha)
axis_Dz_H = one_dim_Dz_u(axis_H)

In [ ]:
def axis_Drr_u(axis_u, u, order=4):
    if order == 4:
        return (-(u[0,:] + u[3,:]) + 81.0 * (u[1,:] + u[2,:]) - 160.0 * axis_u) / (72.0 * (dr / 2.0)**2)
    elif order == 2:
        return (u[0,:] + u[1,:] - 2.0 * axis_u) / (dr / 2.0)**2
    
def axis_Drrrr_u(axis_u, u, order=4):
    if order == 4:
        return (-6.0 * u[4,:] + 65.0 * (u[0,:] + u[3,:]) - 510.0 * (u[1,:] + u[2,:]) + 896.0 * axis_u) / (15.0 * dr**4)

In [ ]:
axis_Drr_alpha = axis_Drr_u(axis_alpha, alpha)
axis_Drr_H = axis_Drr_u(axis_H, H)

In [ ]:
axis_lambda = axis_Drr_H + 2.0 * axis_H * axis_Drr_alpha / axis_alpha + axis_Dz_H * (0.25 * axis_Dz_H / axis_H + axis_Dz_alpha / axis_alpha)

In [ ]:
fig, ax = plt.subplots(figsize = (7, 6))
ax.plot(z[ghost,:], axis_lambda, "-")

ax.set_xlabel(r"$z$")
ax.set_ylabel(r"$\lambda$")
ax.set_title(r"Regularization variable $\lambda$ on axis")

plt.show()

In [ ]:
def one_dim_Dzzz_u(u, order=4, z_sym=1):
    d = np.zeros_like(u)
    if order == 4:
        d[3:-3] = (1.0 / 8.0) * (-13.0 * (u[4:-2] - u[2:-4]) + 8.0 * (u[5:-1] - u[1:-5]) - (u[6:] - u[:-6])) / dz**3
        d[2] = (1.0 / 8.0) * (-13.0 * (u[3] - u[1]) + 8.0 * (u[4] - u[0]) - (u[5] - z_sym * u[4])) / dz**3
        d[0] = -z_sym * d[3]
        d[1] = -z_sym * d[2]
        d[-3] = (1.0 / 8.0) * (       u[-1] +   8.0 * u[-2] -  35.0 * u[-3] +  48.0 * u[-4] -  29.0 * u[-5] +   8.0 * u[-6] -        u[-7]) / dz**3
        d[-2] = (1.0 / 8.0) * (15.0 * u[-1] -  56.0 * u[-2] +  83.0 * u[-3] -  64.0 * u[-4] +  29.0 * u[-5] -   8.0 * u[-6] +        u[-7]) / dz**3
        d[-1] = (1.0 / 8.0) * (49.0 * u[-1] - 232.0 * u[-2] + 461.0 * u[-3] - 496.0 * u[-4] + 307.0 * u[-5] - 104.0 * u[-6] + 15.0 * u[-7]) / dz**3
    elif order == 2:
        d[2:-2] = 0.5 * (-(u[:-4] - u[4:]) + 2.0 * (u[1:-3] - u[3:-1])) / dz**3
        d[1] = 0.5 * (-z_sym * u[2] + u[3] + 2.0 * u[0] - 2.0 * u[2]) / dz**3
        d[0] = -z_sym * d[1]
        d[-2] = 0.5 * (3.0 * u[-1] - 10.0 * u[-2] + 12.0 * u[-3] -  6.0 * u[-4] +       u[-5]) / dz**3
        d[-1] = 0.5 * (5.0 * u[-1] - 18.0 * u[-2] + 24.0 * u[-3] - 14.0 * u[-4] + 3.0 * u[-5]) / dz**3
    return d

In [ ]:
H = np.exp(2.0 * log_h_f)
Dr_H = Dr_u(H)
Dz_H = Dz_u(H)
Drr_H = Drr_u(H)
Dzz_H = Dzz_u(H)

In [ ]:
alpha = np.exp(log_alpha_f)
Dr_alpha = Dr_u(alpha)
Dz_alpha = Dz_u(alpha)
Drr_alpha = Drr_u(alpha)
Dzz_alpha = Dzz_u(alpha)

In [ ]:
omega = beta_f
Dr_omega = Dr_u(omega)
Drr_omega = Drr_u(omega)

In [ ]:
psi = psi_f
Dr_psi = Dr_u(psi)
Drr_psi = Drr_u(psi)

In [ ]:
H00 = axis_i(H, Dr_H, Drr_H)
H01 = one_dim_Dz_u (H00)
H02 = one_dim_Dzz_u(H00)
H20 = axis_Drr_u(H00, H)
H21 = axis_Drr_u(H01, Dz_H)
H22 = axis_Drr_u(H02, Dzz_H)
H03 = one_dim_Dzzz_u(H00)
H40 = axis_Drrrr_u(H00, H)

In [ ]:
alpha00 = axis_i(alpha, Dr_alpha, Drr_alpha)
alpha01 = one_dim_Dz_u (alpha00)
alpha02 = one_dim_Dzz_u(alpha00)
alpha20 = axis_Drr_u(alpha00, alpha)
alpha21 = axis_Drr_u(alpha01, Dz_alpha)
alpha22 = axis_Drr_u(alpha02, Dzz_alpha)
alpha03 = one_dim_Dzzz_u(alpha00)
alpha40 = axis_Drrrr_u(alpha00, alpha)

In [ ]:
omega00 = axis_i(omega, Dr_omega, Drr_omega)
omega01 = one_dim_Dz_u (omega00)

In [ ]:
psi00 = axis_i(psi, Dr_psi, Drr_psi)

In [ ]:
fig, ax = plt.subplots(figsize = (7, 6))


ax.plot(z[ghost,:64], H01[:64], label = r"$\partial_z H$")
ax.plot(z[ghost,:64], H02[:64], label = r"$\partial_{zz} H$")
ax.plot(z[ghost,:64], H20[:64], label = r"$\partial_{\rho \rho} H$")
ax.plot(z[ghost,:64], H21[:64], label = r"$\partial_{\rho \rho z} H$")
ax.plot(z[ghost,:64], H22[:64], label = r"$\partial_{\rho \rho z z} H$")
ax.plot(z[ghost,:64], H03[:64], label = r"$\partial_{zzz} H$")
ax.plot(z[ghost,:64], H40[:64], label = r"$\partial_{\rho \rho \rho \rho} H$")

#ax.plot(z[ghost,:], np.abs(A[ghost,:] - H[ghost,:] + (9.0 * dr**2 * axis_lambda / 4.0)), label = r"$\epsilon = |A - H - \rho^2 \lambda|$")

ax.set_xlabel(r"$z$")
ax.set_ylabel(r"$\lambda$")
ax.legend()
ax.set_title(r"Metric $H$ derivatives")

plt.show()

In [ ]:
foo_1 = (-H01**4/(6.*H00**3) + (3*H01**2*H02)/(8.*H00**2) - H02**2/(12.*H00) - 
   (H01*H03)/(12.*H00) + (7*H01**2*H20)/(24.*H00**2) + (17*H20**2)/(12.*H00) + (5*H01*H21)/(12.*H00) - 
   H22/6. + H40/18.)

foo_2 = ((5*H01**3*alpha01)/(24.*H00**2*alpha00) + (H01*H02*alpha01)/(3.*H00*alpha00) - (H03*alpha01)/(6.*alpha00) + 
   (13*H01*H20*alpha01)/(6.*H00*alpha00) + (H21*alpha01)/(6.*alpha00) + (H01**2*alpha01**2)/(2.*H00*alpha00**2) + 
   (H02*alpha01**2)/(3.*alpha00**2) - (H01*alpha01**3)/(3.*alpha00**3) + (H01**2*alpha02)/(3.*H00*alpha00) - 
   (H02*alpha02)/(3.*alpha00) + (H01*alpha01*alpha02)/(2.*alpha00**2) - (H01*alpha03)/(6.*alpha00) + (13*H01**2*alpha20)/(12.*H00*alpha00) - 
   (H02*alpha20)/(3.*alpha00) + (5*H20*alpha20)/alpha00 + (19*H01*alpha01*alpha20)/(6.*alpha00**2) - 
   (2*H00*alpha01**2*alpha20)/(3.*alpha00**3) + (H00*alpha02*alpha20)/(3.*alpha00**2) + (3*H00*alpha20**2)/alpha00**2 + 
   (H01*alpha21)/(6.*alpha00) + (2*H00*alpha01*alpha21)/(3.*alpha00**2) - (H00*alpha22)/(3.*alpha00) + (H00*alpha40)/(9.*alpha00) + 
   (H00**2*omega01**2)/(4.*alpha00**2))

foo_3 = (32*H00*np.pi*psi00**2)/3.

In [ ]:
axis_Drr_lambda = foo_1 + foo_2 + foo_3

In [ ]:
i = ghost

In [ ]:
fig, ax = plt.subplots(figsize = (7, 6))
ax.plot(r[:128,i], axis_lambda[i] + 0.5 * axis_Drr_lambda[i] * r[:128,i]**2, label = r"Taylor Series")
ax.plot(r[:128,i], ((A - H)/r**2)[:128,i], label = r"$(A-H)/\rho^2$")
ax.plot(r[:128,i], lambda_f[:128,i], label = r"$\lambda$")

#ax.plot(r[:128,i], np.log10(np.abs(-lambda_f[:,i] + axis_lambda[i] + 0.5 * axis_Drr_lambda[i] * r[:,i]**2))[:128], label = r"Taylor Series")
#ax.plot(r[:128,i], np.log10(np.abs(-lambda_f + (A - H)/r**2))[:128,i], label = r"$(A-H)/\rho^2$")

ax.legend()
ax.set_ylim(1.05 * np.min(lambda_f[:128,i]), 0.005)

ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$\lambda$")
ax.set_title(r"Regularization $\lambda_f$ profile on equator")

plt.show()

# Interpolation

In [ ]:
#np.savetxt("/mnt/d/ROTBOSON/out/l=2,w=7.00000E-01,dr=6.25000E-02,N=0512,IRREGULAR/log_alpha_i.asc", log_alpha_f[1:-1,1:-1])
#np.savetxt("/mnt/d/ROTBOSON/out/l=2,w=7.00000E-01,dr=6.25000E-02,N=0512,IRREGULAR/beta_i.asc", beta_f[1:-1,1:-1])
#np.savetxt("/mnt/d/ROTBOSON/out/l=2,w=7.00000E-01,dr=6.25000E-02,N=0512,IRREGULAR/log_h_i.asc", log_h_f[1:-1,1:-1])
#np.savetxt("/mnt/d/ROTBOSON/out/l=2,w=7.00000E-01,dr=6.25000E-02,N=0512,IRREGULAR/log_a_i.asc", log_a_f[1:-1,1:-1])
#np.savetxt("/mnt/d/ROTBOSON/out/l=2,w=7.00000E-01,dr=6.25000E-02,N=0512,IRREGULAR/psi_i.asc", psi_f[1:-1,1:-1])
#np.savetxt("/mnt/d/ROTBOSON/out/l=2,w=7.00000E-01,dr=6.25000E-02,N=0512,IRREGULAR/lambda_i.asc", lambda_f[1:-1,1:-1])

In [ ]:
#ghost = 2
#rr_inf = (r.shape[0] - 2 * ghost) * dr

In [ ]:
#interior_ns = np.round(np.logspace(np.log10(128), np.log10(1024), 8)).astype(int)
#interior_ns = np.round(np.arange(128, 1024 + 1, 128)).astype(int)
#print(interior_ns)
#resolutions = rr_inf / interior_ns
#print(resolutions)

In [ ]:
#interp_log_alpha_function = interpolate.RectBivariateSpline(r[:,0], z[0,:], log_alpha_f)
#interp_beta_function = interpolate.RectBivariateSpline(r[:,0], z[0,:], beta_f)
#interp_log_h_function = interpolate.RectBivariateSpline(r[:,0], z[0,:], log_h_f)
#interp_log_a_function = interpolate.RectBivariateSpline(r[:,0], z[0,:], log_a_f)
#interp_psi_function = interpolate.RectBivariateSpline(r[:,0], z[0,:], psi_f)

In [ ]:
#j = 7
#print(interior_ns[j], resolutions[j])

In [ ]:
#for j in range(interior_ns.size):
#max_resolution_r = np.linspace(resolutions[j] * (0.5 - ghost), resolutions[j] * (interior_ns[j] - 0.5 + ghost), interior_ns[j] + 2 * ghost)
#max_resolution_z = np.linspace(resolutions[j] * (0.5 - ghost), resolutions[j] * (interior_ns[j] - 0.5 + ghost), interior_ns[j] + 2 * ghost)

#max_mesh_z, max_mesh_r = np.meshgrid(max_resolution_z, max_resolution_r)

#max_resolution_log_alpha = interp_log_alpha_function(max_resolution_r, max_resolution_z)
#max_resolution_beta = interp_beta_function(max_resolution_r, max_resolution_z)
#max_resolution_log_h = interp_log_h_function(max_resolution_r, max_resolution_z)
#max_resolution_log_a = interp_log_a_function(max_resolution_r, max_resolution_z)
#max_resolution_psi = interp_psi_function(max_resolution_r, max_resolution_z)

#np.savetxt("IIData/log_alpha%d.asc" % j, max_resolution_log_alpha)
#np.savetxt("IIData/beta%d.asc" % j, max_resolution_beta)
#np.savetxt("IIData/log_h%d.asc" % j, max_resolution_log_h)
#np.savetxt("IIData/log_a%d.asc" % j, max_resolution_log_a)
#np.savetxt("IIData/psi%d.asc" % j, max_resolution_psi)

In [ ]:
#np.savetxt("RIData/log_alpha0.asc", log_alpha[:interior_ns[0] + 2 * ghost, :interior_ns[0] + 2 * ghost])
#np.savetxt("RIData/beta0.asc", beta[:interior_ns[0] + 2 * ghost, :interior_ns[0] + 2 * ghost])
#np.savetxt("RIData/log_h0.asc", log_h[:interior_ns[0] + 2 * ghost, :interior_ns[0] + 2 * ghost])
#np.savetxt("RIData/log_a0.asc", log_a[:interior_ns[0] + 2 * ghost, :interior_ns[0] + 2 * ghost])
#np.savetxt("RIData/psi0.asc", psi[:interior_ns[0] + 2 * ghost, :interior_ns[0] + 2 * ghost])
#
#np.savetxt("RIData/log_alpha1.asc", log_alpha[:interior_ns[1] + 2 * ghost, :interior_ns[1] + 2 * ghost])
#np.savetxt("RIData/beta1.asc", beta[:interior_ns[1] + 2 * ghost, :interior_ns[1] + 2 * ghost])
#np.savetxt("RIData/log_h1.asc", log_h[:interior_ns[1] + 2 * ghost, :interior_ns[1] + 2 * ghost])
#np.savetxt("RIData/log_a1.asc", log_a[:interior_ns[1] + 2 * ghost, :interior_ns[1] + 2 * ghost])
#np.savetxt("RIData/psi1.asc", psi[:interior_ns[1] + 2 * ghost, :interior_ns[1] + 2 * ghost])
#
#np.savetxt("RIData/log_alpha2.asc", log_alpha[:interior_ns[2] + 2 * ghost, :interior_ns[2] + 2 * ghost])
#np.savetxt("RIData/beta2.asc", beta[:interior_ns[2] + 2 * ghost, :interior_ns[2] + 2 * ghost])
#np.savetxt("RIData/log_h2.asc", log_h[:interior_ns[2] + 2 * ghost, :interior_ns[2] + 2 * ghost])
#np.savetxt("RIData/log_a2.asc", log_a[:interior_ns[2] + 2 * ghost, :interior_ns[2] + 2 * ghost])
#np.savetxt("RIData/psi2.asc", psi[:interior_ns[2] + 2 * ghost, :interior_ns[2] + 2 * ghost])
#
#np.savetxt("RIData/log_alpha3.asc", log_alpha[:interior_ns[3] + 2 * ghost, :interior_ns[3] + 2 * ghost])
#np.savetxt("RIData/beta3.asc", beta[:interior_ns[3] + 2 * ghost, :interior_ns[3] + 2 * ghost])
#np.savetxt("RIData/log_h3.asc", log_h[:interior_ns[3] + 2 * ghost, :interior_ns[3] + 2 * ghost])
#np.savetxt("RIData/log_a3.asc", log_a[:interior_ns[3] + 2 * ghost, :interior_ns[3] + 2 * ghost])
#np.savetxt("RIData/psi3.asc", psi[:interior_ns[3] + 2 * ghost, :interior_ns[3] + 2 * ghost])
#
#np.savetxt("RIData/log_alpha4.asc", log_alpha[:interior_ns[4] + 2 * ghost, :interior_ns[4] + 2 * ghost])
#np.savetxt("RIData/beta4.asc", beta[:interior_ns[4] + 2 * ghost, :interior_ns[4] + 2 * ghost])
#np.savetxt("RIData/log_h4.asc", log_h[:interior_ns[4] + 2 * ghost, :interior_ns[4] + 2 * ghost])
#np.savetxt("RIData/log_a4.asc", log_a[:interior_ns[4] + 2 * ghost, :interior_ns[4] + 2 * ghost])
#np.savetxt("RIData/psi4.asc", psi[:interior_ns[4] + 2 * ghost, :interior_ns[4] + 2 * ghost])
#
#np.savetxt("RIData/log_alpha5.asc", log_alpha[:interior_ns[5] + 2 * ghost, :interior_ns[5] + 2 * ghost])
#np.savetxt("RIData/beta5.asc", beta[:interior_ns[5] + 2 * ghost, :interior_ns[5] + 2 * ghost])
#np.savetxt("RIData/log_h5.asc", log_h[:interior_ns[5] + 2 * ghost, :interior_ns[5] + 2 * ghost])
#np.savetxt("RIData/log_a5.asc", log_a[:interior_ns[5] + 2 * ghost, :interior_ns[5] + 2 * ghost])
#np.savetxt("RIData/psi5.asc", psi[:interior_ns[5] + 2 * ghost, :interior_ns[5] + 2 * ghost])
#
#np.savetxt("RIData/log_alpha6.asc", log_alpha[:interior_ns[6] + 2 * ghost, :interior_ns[6] + 2 * ghost])
#np.savetxt("RIData/beta6.asc", beta[:interior_ns[6] + 2 * ghost, :interior_ns[6] + 2 * ghost])
#np.savetxt("RIData/log_h6.asc", log_h[:interior_ns[6] + 2 * ghost, :interior_ns[6] + 2 * ghost])
#np.savetxt("RIData/log_a6.asc", log_a[:interior_ns[6] + 2 * ghost, :interior_ns[6] + 2 * ghost])
#np.savetxt("RIData/psi6.asc", psi[:interior_ns[6] + 2 * ghost, :interior_ns[6] + 2 * ghost])

## Angular Derivatives

In [ ]:
plot_2d(1, r_ex, z_ex, np.abs(z * Dr_u(log_alpha_f) - r * Dz_u(log_alpha_f)), r"$\partial_\theta\,\log\alpha$")

In [ ]:
fig, ax = plt.subplots()
ax.plot(r[:,2], np.log10(np.abs(z * Dr_u(log_alpha_f)/log_alpha_f - r * Dz_u(log_alpha_f)/log_alpha_f))[:,2], label = r"$\rho$")
ax.plot(z[2,:], np.log10(np.abs(z * Dr_u(log_alpha_f)/log_alpha_f - r * Dz_u(log_alpha_f)/log_alpha_f))[2,:], label = r"$z$")
ax.plot(np.diag(np.sqrt(r**2 + z**2)), np.diag(np.log10(np.abs(z * Dr_u(log_alpha_f)/log_alpha_f - r * Dz_u(log_alpha_f)/log_alpha_f))), label = r"$d$")
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\partial_\theta\,\log\,\alpha$")
ax.legend()
plt.show()